# 02 · Data Cleaning

**Project:** Data Analyst – Mental Health (Canada) · **Pipeline step:** 2 of 10

> **Skeleton only.** This notebook currently just wires up the libraries and the
> `data/raw` → `data/processed` connections so the team can start cleaning tomorrow.
> The cleaning logic goes under section 4.


## 1 · Libraries

In [12]:
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
print("pandas", pd.__version__, "| numpy", np.__version__)


pandas 3.0.5 | numpy 2.5.2


## 2 · Paths (`data/raw` → `data/processed`)

In [13]:
def find_root(start: Path) -> Path:
    """Walk up until we find the folder that contains data/raw (works from repo root or /notebooks)."""
    for p in [start, *start.parents]:
        if (p / "data" / "raw").is_dir():
            return p
    raise FileNotFoundError("Could not find data/raw above " + str(start))

ROOT = find_root(Path.cwd())
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

print("root     :", ROOT)
print("raw      :", RAW)
print("processed:", PROCESSED)


root     : /home/codespace/createprojectfolder/Data-Analyst-Mental-Health-Project
raw      : /home/codespace/createprojectfolder/Data-Analyst-Mental-Health-Project/data/raw
processed: /home/codespace/createprojectfolder/Data-Analyst-Mental-Health-Project/data/processed


## 3 · Load the raw datasets

Same registry as `01_data_understanding.ipynb`. StatCan CSVs need `utf-8-sig` (BOM).
The CIHI Excel workbook is loaded separately (only the two hidden data sheets are usable).

In [15]:
DATASETS = {
    "perceived_mh_annual":        {"file": "StatCan 13-10-0972 – perceived mental health.csv",                                              "kind": "statcan_long"},
    "suicidal_thoughts":          {"file": "Catalogue Entry Mental health characteristics and suicidal thoughts.csv",                        "kind": "statcan_long"},
    "stress_coping":              {"file": "Catalogue Entry Mental health characteristics Ability to handle stress and sources of stress.csv","kind": "statcan_long"},
    "perceived_health_quarterly": {"file": "Catalogue Entry Mental health indicators.csv",                                                   "kind": "statcan_long"},
    "cchs_mh_disorders":          {"file": "Catalogue Entry Perceived health, by gender and province.csv",                                   "kind": "statcan_long"},
    "cihi_mh_services":           {"file": "health services for mental illness and alcoholdrug induced disorders.csv",                       "kind": "cihi_vizconfig"},
    "cihi_children_youth":        {"file": "care-children-youth-with-mental-disorders-data-tables-en.xlsx",                                  "kind": "excel_multitable"},
    "mhacs_2022_pumf":            {"file": "MHACS 2022 Public Use Microdata.csv",                                                            "kind": "microdata"},
}

def load_dataset(key: str) -> pd.DataFrame:
    spec = DATASETS[key]
    path = RAW / spec["file"]
    if spec["kind"] in ("statcan_long", "cihi_vizconfig"):
        return pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
    if spec["kind"] == "microdata":
        return pd.read_csv(path, low_memory=False)
    raise ValueError(f"{key}: kind={spec['kind']} is loaded separately (see below)")

# CSV datasets -> raw[...]
raw = {k: load_dataset(k) for k, v in DATASETS.items() if v["kind"] != "excel_multitable"}

# CIHI workbook: the two machine-readable sheets (title row 0, headers row 1)
_xls = pd.ExcelFile(RAW / DATASETS["cihi_children_youth"]["file"])
raw_excel = {}
for _sheet in [s for s in _xls.sheet_names if s.endswith("_to hide")]:
    _t = _xls.parse(_sheet, header=None)
    _body = _t.iloc[2:].reset_index(drop=True)
    _body.columns = [str(h).replace("\n", " ").strip() for h in _t.iloc[1]]
    raw_excel[_sheet] = _body

for k, df in raw.items():
    print(f"{k:28} {df.shape}")
for k, df in raw_excel.items():
    print(f"{k:28} {df.shape}  (excel)")


perceived_mh_annual          (936, 18)
suicidal_thoughts            (8208, 18)
stress_coping                (27360, 18)
perceived_health_quarterly   (6318, 17)
cchs_mh_disorders            (160992, 18)
cihi_mh_services             (264, 14)
mhacs_2022_pumf              (9861, 602)
Table8DATA_to hide           (216, 13)  (excel)
Table13DATA_to hide          (216, 13)  (excel)


## 4 · Cleaning — TO BE COMPLETED BY THE TEAM

Write cleaned outputs to `PROCESSED / "..."`. Suggested tasks (see `docs/data_dictionary.md` §A):

- [ ] StatCan tables: melt to tidy long; pivot `Characteristics`/`Statistics` into `value` / `ci_low` / `ci_high` / `cv`
- [ ] Standardise `GEO` to one canonical province list; handle region rollups separately
- [ ] Parse `REF_DATE` (year / year-range / year-month) into a real date/period
- [ ] Apply `SCALAR_FACTOR` (×1000 where `thousands`); keep `STATUS` as `quality_flag`; **do not impute** suppressed values
- [ ] Filter to the agreed analysis window
- [ ] `cihi_mh_services`: unpivot chart-config rows → `indicator | breakdown | group | value | ci_low | ci_high`
- [ ] `cihi_children_youth`: split each `95% CI` string into `ci_low` / `ci_high`; tidy long
- [ ] `mhacs_2022_pumf`: replace non-response codes (6/7/8/9, 96, 996, 99.6 …) with NaN for the selected variables only
- [ ] Save each cleaned dataset to `data/processed/` and note row counts


In [13]:
def clean_statcan_long(df: pd.DataFrame) -> pd.DataFrame:
    """Clean StatCan long-format data while retaining characteristic-level observations."""
    df = df.copy()

    # Normalize columns: lowercase, strip whitespace
    df.columns = df.columns.str.strip().str.lower()
    text_cols = df.select_dtypes(include="object").columns
    for col in text_cols:
        df[col] = df[col].str.strip()

    # Standardize column names
    rename_map = {
        "age group": "age_group", "indicators": "indicator", "characteristics": "characteristic",
        "statistics": "statistic", "ref_date": "ref_date", "value": "value", "dguid": "dguid",
        "uom_id": "uom_id", "scalar_factor": "scalar_factor", "scalar_id": "scalar_id",
        "vector": "vector", "coordinate": "coordinate", "status": "quality_flag"
    }
    df.rename(columns=rename_map, inplace=True)
    if "gender" in df.columns:
        df.rename(columns={"gender": "sex"}, inplace=True)

    # Parse reference dates: preserve original format, extract year for sorting
    df["ref_date_raw"] = df["ref_date"].astype(str).str.strip()
    df["start_year"] = pd.to_numeric(
        df["ref_date_raw"].str.extract(r"^(\d{4})")[0], errors="coerce"
    ).astype("Int64")

    # Convert value to numeric, apply scalar factors
    if "value" in df.columns:
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
    if "scalar_factor" in df.columns and "value" in df.columns:
        df.loc[df["scalar_factor"].eq("thousands"), "value"] *= 1000
        df["scalar_applied"] = df["scalar_factor"].eq("thousands")

    # Normalize quality flags and metric types
    if "quality_flag" in df.columns:
        df["quality_flag"] = df["quality_flag"].str.lower().str.strip()
    if "uom" in df.columns:
        df["metric_type"] = df["uom"].str.lower()

    # Drop metadata columns
    drop_cols = ["symbol", "terminated", "decimals", "uom_id", "scalar_id", "coordinate"]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

    # Normalize geography and indicators
    if "geo" in df.columns:
        df["geo"] = df["geo"].str.replace(" / ", "/", regex=False).str.strip()
    if "indicator" in df.columns:
        df["indicator"] = df["indicator"].str.replace(r"\s+", " ", regex=True).str.strip()

    # Keep CI observations in long form until their grouping keys are validated.
    col_order = ["ref_date_raw", "start_year", "geo", "sex", "age_group", "indicator", 
                 "characteristic", "statistic", "value", "metric_type", "quality_flag", 
                 "scalar_applied", "vector"]
    cols_present = [c for c in col_order if c in df.columns]
    other_cols = [c for c in df.columns if c not in cols_present]
    df = df[cols_present + other_cols]

    return df

In [8]:
def clean_cihi_vizconfig(df: pd.DataFrame) -> pd.DataFrame:
    """Unpivot CIHI chart-config CSV: split x/y axis values into separate rows."""
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower()
    df["x_axis_values"] = df["x_axis_values"].astype(str).str.split(",")
    df["y_axis_values"] = df["y_axis_values"].astype(str).str.split(",")

    tidy_rows = []
    for _, row in df.iterrows():
        indicator = row.get("indicator", "")
        xs = row["x_axis_values"]
        ys = row["y_axis_values"]
        n = min(len(xs), len(ys))

        for i in range(n):
            x = xs[i].strip()
            y = ys[i].strip()
            value = pd.to_numeric(y, errors="coerce")
            tidy_rows.append({
                "indicator": indicator,
                "breakdown": row.get("vis_option", None),  # FIXED: was "breakdown" (doesn't exist)
                "group": x,
                "value": value,
                "ci_low": pd.to_numeric(row.get("confidence_interval_low", None), errors="coerce"),  # FIXED: was "ci_low"
                "ci_high": pd.to_numeric(row.get("confidence_interval_high", None), errors="coerce"),  # FIXED: was "ci_high"
            })

    tidy = pd.DataFrame(tidy_rows)
    tidy = tidy.dropna(subset=["value"])
    return tidy

In [9]:
def clean_cihi_children_youth(raw_excel: dict) -> pd.DataFrame:
    """Reshape CIHI children/youth Excel from wide to long, parse CI bounds, add sheet identifier."""
    frames = []
    for sheet, df in raw_excel.items():
        df = df.copy()
        df.columns = df.columns.str.strip().str.replace("\n", " ")

        # Identify year columns and ID columns
        year_cols = [c for c in df.columns if any(y in str(c) for y in ["2018", "2019", "2020", "2021", "2022"])]
        id_cols = [c for c in df.columns if c not in year_cols] if year_cols else list(df.columns[:3])
        value_cols = year_cols if year_cols else list(df.columns[3:])

        # Melt to long format
        long_df = df.melt(id_vars=id_cols, value_vars=value_cols, var_name="fiscal_year", value_name="value")
        long_df["fiscal_year"] = long_df["fiscal_year"].str.replace(" ", "", regex=False)

        # Parse CI from value strings (e.g., "12.3-14.5" → ci_low, ci_high)
        long_df["ci_low"] = None
        long_df["ci_high"] = None
        ci_pattern = r"([\d.]+)\s*-\s*([\d.]+)"
        for idx, val in enumerate(long_df["value"]):
            if isinstance(val, str) and "-" in val:
                match = pd.Series(val).str.extract(ci_pattern, expand=True)
                if not match.empty and match.iloc[0, 0] is not None:
                    long_df.loc[idx, "ci_low"] = pd.to_numeric(match.iloc[0, 0], errors="coerce")
                    long_df.loc[idx, "ci_high"] = pd.to_numeric(match.iloc[0, 1], errors="coerce")
        long_df["value"] = pd.to_numeric(long_df["value"], errors="coerce")

        # Add sheet type identifier
        long_df["sheet_type"] = "ED" if "Table8" in sheet else ("Hospitalization" if "Table13" in sheet else sheet.replace("_to hide", ""))
        frames.append(long_df)

    return pd.concat(frames, ignore_index=True)

In [10]:
def clean_mhacs_pumf(df: pd.DataFrame) -> pd.DataFrame:
    """Clean MHACS 2022 PUMF: replace missing codes only for documented behavioral/health variables."""
    df = df.copy()
    
    # Only replace missing codes for specific variables (not all 602 columns)
    documented_variables = {"DMHSTAT", "DPHSTAT", "DALCOHOL", "DDRUGS", "DWKDECAL", "DWKDISAB"}
    df_cols_lower = {c.lower(): c for c in df.columns}
    target_vars = [df_cols_lower.get(v.lower(), v) for v in documented_variables if v.lower() in df_cols_lower or v in df.columns]

    # Include any column starting with D that is numeric
    for col in df.columns:
        if col.startswith("D") and df[col].dtype in ["int64", "float64"] and col not in target_vars:
            target_vars.append(col)

    # Replace missing codes only in target variables
    missing_codes = {6, 7, 8, 9, 96, 996, 999, 99.6}
    for col in target_vars:
        if col in df.columns:
            df[col] = df[col].replace(list(missing_codes), np.nan)

    # Ensure weight is numeric
    if "WTS_M" in df.columns:
        df["WTS_M"] = pd.to_numeric(df["WTS_M"], errors="coerce")

    return df

In [16]:
CLEANED = ROOT / "data" / "processed" / "02_cleaned" # Builds a path object pointing to the folder where you want to store all cleaned datasets
CLEANED.mkdir(parents=True, exist_ok=True) # Creates that folder if it doesn’t exist

cleaned = {} # Creates an empty dictionary that will store cleaned DataFrames keyed by dataset name, e.g.: 
             # cleaned["perceived_mh_annual"], cleaned["cihi_mh_services"], cleaned["mhacs_2022_pumf"]

for key, spec in DATASETS.items():
    print(f"\n=== Cleaning {key} ===")

    if spec["kind"] == "statcan_long": # Calls your clean_statcan_long function on the raw DataFrame:
        df = clean_statcan_long(raw[key])   # raw[key] is the raw version loaded earlier
        cleaned[key] = df                     # Stores the cleaned DataFrame in the cleaned dict
        df.to_csv(CLEANED / f"{key}.csv", index=False) # Writes the cleaned DataFrame to disk as a CSV: data/processed/02_cleaned

    elif spec["kind"] == "cihi_vizconfig":  # Same process for rest all
        df = clean_cihi_vizconfig(raw[key])
        cleaned[key] = df
        df.to_csv(CLEANED / f"{key}.csv", index=False)

    elif spec["kind"] == "excel_multitable":
        df = clean_cihi_children_youth(raw_excel)
        cleaned[key] = df
        df.to_csv(CLEANED / f"{key}.csv", index=False)

    elif spec["kind"] == "microdata":
        df = clean_mhacs_pumf(raw[key])
        cleaned[key] = df
        df.to_csv(CLEANED / f"{key}.csv", index=False)

print("\nALL DATASETS CLEANED ✔") # Prints final msg



=== Cleaning perceived_mh_annual ===

=== Cleaning suicidal_thoughts ===

=== Cleaning stress_coping ===

=== Cleaning perceived_health_quarterly ===

=== Cleaning cchs_mh_disorders ===

=== Cleaning cihi_mh_services ===

=== Cleaning cihi_children_youth ===

=== Cleaning mhacs_2022_pumf ===

ALL DATASETS CLEANED ✔
